# Data Preprocessing Overview

This notebook builds the model-ready datasets from `./results/df.csv` (written by `create_data.ipynb`). It is designed to be run **ten times**, once per `SEED` value from 1 to 10, producing ten dataset variants for the robustness analysis.

## Prerequisite
Run `create_data.ipynb` first - it writes `./results/df.csv`.

## What this file produces
- **Missingness-aware matching (missing-matched):** healthy controls are matched to each AD/dementia case by their pattern of missing features. This is the primary cohort used in the paper, and an **age-matched** version is produced alongside it.
- Each cohort is written in three versions:
  - **unimputed** - missing values kept (XGBoost handles them natively);
  - **KNN-imputed** - custom cosine-distance nearest-neighbour imputation;
  - **RCS-imputed** - Random Case Sampling: each missing value is filled with a randomly drawn observed value from a balanced reference pool.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
from tqdm import tqdm

# set that seed to 1 to 10
SEED = 10
random.seed(41 + SEED)
np.random.seed(41 + SEED)

# Functions

In [ ]:
def randomfill_missing_values(df, columns_with_missing_values, number_of_samples=100):
    for col in columns_with_missing_values:
        # calculate mean of random 100 samples for each status group
        statuses = df["Status"].unique()
        samples = []
        for status in statuses:
            # get a random sample of 100 rows for this Status
            sample = df[df["Status"] == status][col].dropna()
            if len(sample) > number_of_samples:
                sample = sample.sample(
                    n=number_of_samples, random_state=41 + SEED
                )  # randomly sample 100 rows
            # combine samples for each status
            samples.extend(sample)

        samples = pd.Series(samples)

        # get the missing indices
        missing_indices = df[col].isna()

        # num of missing values to this
        num_missing = missing_indices.sum()

        if num_missing > 0:
            # randomly select values from samples
            random_values = random.choices(samples.tolist(), k=num_missing)

            # fill the missing values with random choices
            random_series = pd.Series(random_values, index=df[missing_indices].index)
            df.loc[missing_indices, col] = random_series

        print(f"col: {col}, filled {num_missing} missing values with random samples")

    # check remaining missing values
    print(df.isna().sum())
    return df


def normalized_euclidean_distance(row, candidates, NUM_FEATURES):
    # Select numeric features for the current row and all candidate rows
    row_numeric = row[NUM_FEATURES]
    candidates_numeric = candidates[NUM_FEATURES]

    # Calculate the difference only for non-missing values
    valid_mask = ~pd.isna(row_numeric)
    differences = candidates_numeric.loc[:, valid_mask] - row_numeric[valid_mask]

    squared_diff = np.sum(differences**2, axis=1)
    normalization_factor = valid_mask.sum() if valid_mask.sum() != 0 else 1
    distances = (squared_diff ** (1 / 2)) / normalization_factor
    return distances


def cosine_similarity(row, candidates, NUM_FEATURES):
    # Select numeric features for the current row and all candidate rows
    row_numeric = row[NUM_FEATURES]
    candidates_numeric = candidates[NUM_FEATURES]

    # Calculate the dot product only for non-missing values
    valid_mask = ~pd.isna(row_numeric)
    dot_product = np.dot(candidates_numeric.loc[:, valid_mask], row_numeric[valid_mask])

    # Calculate the norms
    row_norm = np.linalg.norm(row_numeric[valid_mask])
    candidates_norm = np.linalg.norm(candidates_numeric.loc[:, valid_mask], axis=1)

    # Calculate cosine similarity and convert to cosine distance
    cosine_sim = dot_product / (row_norm * candidates_norm + 1e-9)
    cosine_distance = 1 - cosine_sim

    cosine_distance_series = pd.Series(cosine_distance, index=candidates.index)
    return cosine_distance_series


def custom_knn_impute_euclidean(
    X,
    k,
    NUM_FEATURES,
    categorical_cols,
    columns_with_missing_values,
    cat_func="mode",
    distance_func=cosine_similarity,
):
    X_imputed = X.copy()

    # Use tqdm to wrap the iterrows generator with a progress bar
    for idx, row in tqdm(X.iterrows(), total=len(X), desc="Imputing rows"):
        # skip the rows without missing values
        if not row.isnull().any():
            continue

        # columns current row is missing
        null_mask = row.isnull()
        # numerical columns mask
        num_mask = X.columns.isin(NUM_FEATURES)

        combined_mask = null_mask | num_mask
        candidate_mask = X.loc[:, combined_mask].notna().all(axis=1)

        candidate_rows = X.loc[candidate_mask]

        selected_samples = pd.DataFrame()
        # randomly select 100 samples for each status
        for s in candidate_rows["Status"].unique():
            current_group = candidate_rows[candidate_rows["Status"] == s]
            # if there are more than 100 sample randomly select 100, else select all of them
            number_of_samples = 100 if len(current_group) >= 100 else len(current_group)
            selected_samples = pd.concat(
                [
                    selected_samples,
                    current_group.sample(number_of_samples, random_state=41 + SEED),
                ]
            )

        # calculate distances,
        distances = distance_func(row, selected_samples, NUM_FEATURES)
        distances.sort_values(ascending=True, inplace=True)
        # select top k rows
        top_k = distances[:k]
        top_k_rows = selected_samples.loc[top_k.keys()]

        for col in columns_with_missing_values:
            if not pd.isna(row[col]):
                continue
            # perform imputation for missing values
            if col in NUM_FEATURES:
                values = top_k_rows[col]
                imputed_value = np.mean(values)
            elif col in categorical_cols:
                values = top_k_rows[col]
                if cat_func == "median":
                    imputed_value = np.median(values)
                else:
                    imputed_value = values.mode().iloc[0]
            X_imputed.at[idx, col] = imputed_value
    return X_imputed

# Create df and global variables

In [ ]:
df_merged = pd.read_csv(r"./results/df.csv")

output_path = r"./results_{}/".format(SEED)
random.seed(41 + SEED)

os.makedirs(output_path, exist_ok=True)

CAT_FEATURES = [
    "Age",
    "Sex",
    "Educational status",
    "Diabetes",
    "Alcohol Consumption",
    "Smoking Status",
    "Antihypertensive usage",
]
NUM_FEATURES = df_merged.columns.difference(CAT_FEATURES + ["Status", "eid"])
df_merged.isnull().sum()

In [ ]:
# will be used in labeled dataset
df_merged_labeled = df_merged.copy()
# will be used in missing matched dataset
df_missing_matched = df_merged.copy()
# will be used in age matched missing matched dataset
df_missing_matched_age_matched = df_merged.copy()

df_full = df_merged.dropna(axis=1).copy()

columns_with_missing_values = df_merged.columns[df_merged.isna().sum() > 0]

In [ ]:
df_merged.Status.value_counts()

In [ ]:
df_full.info()

# label encoding

In [ ]:
df_merged_labeled["Sex"] = df_merged_labeled["Sex"].replace({0: "Women", 1: "Men"})
df_merged_labeled["Educational status"] = df_merged_labeled[
    "Educational status"
].replace(
    {
        1: "Higher",
        2: "Upper secondary",
        3: "Lower secondary",
        4: "Lower secondary",
        5: "Vocational",
        6: "Other",
    }
)
df_merged_labeled["Diabetes"] = df_merged_labeled["Diabetes"].replace(
    {0: "Without diabetes", 1: "With diabetes"}
)
df_merged_labeled["Antihypertensive usage"] = df_merged_labeled[
    "Antihypertensive usage"
].replace({0: "Without medication", 1: "With medication"})
df_merged_labeled["Alcohol Consumption"] = df_merged_labeled[
    "Alcohol Consumption"
].replace(
    {
        1: "Daily",
        2: "3 or 4 /week",
        3: "1 or 2/week",
        4: "1-3/month",
        5: "Special occations",
        6: "Never",
    }
)
df_merged_labeled["Smoking Status"] = df_merged_labeled["Smoking Status"].replace(
    {0: "Never", 1: "Previous", 2: "Current"}
)

In [ ]:
df_merged_labeled.to_csv(r"./results/merged_labeled.csv", index=False)

# missing matched


In [ ]:
# helper: compute missing featnature for each row
def get_missing_features(df, cols):
    """Return a Series of tuples indicating which of `cols` are NaN per row."""
    mask = df[cols].isna()
    return mask.apply(lambda row: tuple(sorted(cols[row])), axis=1)

In [ ]:
# build the missing matched datasets
sick_mm = df_missing_matched[
    df_missing_matched["Status"].isin(["dementia", "AD"])
].copy()
healthy_mm = df_missing_matched[df_missing_matched["Status"] == "healthy"].copy()

# compute missing features (only over columns that can have missing values)
sick_mm["_feat"] = get_missing_features(sick_mm, columns_with_missing_values)
healthy_mm["_feat"] = get_missing_features(healthy_mm, columns_with_missing_values)

matched_healthy_list = []

for feat, group in sick_mm.groupby("_feat"):
    count = len(group)
    # no missing  → 10 healthy matches per sick;  has missing → 1 healthy per sick
    needed = count * 10 if len(feat) == 0 else count * 1
    candidates = healthy_mm[healthy_mm["_feat"] == feat]

    if len(candidates) >= needed:
        sampled = candidates.sample(n=needed, random_state=41 + SEED)
    else:
        sampled = candidates
        print(
            f"Warning (missing-matched): Only {len(candidates)} healthy candidates "
            f"for pattern {list(feat) if feat else 'no-missing'}, needed {needed}"
        )

    matched_healthy_list.append(sampled)

matched_healthy_mm = pd.concat(matched_healthy_list, ignore_index=True)

# drop helper column
sick_mm = sick_mm.drop(columns=["_feat"])
matched_healthy_mm = matched_healthy_mm.drop(columns=["_feat"])

# combine and shuffle
df_mm = pd.concat([sick_mm, matched_healthy_mm], ignore_index=True)
df_mm = df_mm.sample(frac=1, random_state=41 + SEED).reset_index(drop=True)

print("\nMissing-Matched dataset distribution:")
print(df_mm["Status"].value_counts())

In [ ]:
df_missing_matched.isnull().sum()

In [ ]:
df_mm[df_mm["Status"] == "AD"].isnull().sum()

In [ ]:
df_mm_missing = df_mm.copy()
df_mm_randomfill = df_mm.copy()
df_mm_knn = df_mm.copy()

df_mm_missing.to_csv(output_path + r"missing_matched_missing.csv", index=False)

df_mm_randomfill = randomfill_missing_values(
    df_mm_randomfill, columns_with_missing_values
)
df_mm_randomfill.to_csv(output_path + r"missing_matched_randomfill.csv", index=False)

df_mm_knn = custom_knn_impute_euclidean(
    df_mm_knn,
    k=5,
    NUM_FEATURES=NUM_FEATURES,
    categorical_cols=CAT_FEATURES,
    columns_with_missing_values=columns_with_missing_values,
    distance_func=cosine_similarity,
)
df_mm_knn.to_csv(output_path + r"missing_matched_knn.csv", index=False)

In [ ]:
# build the age + missing-matched dataset
sick_amm = df_missing_matched_age_matched[
    df_missing_matched_age_matched["Status"].isin(["dementia", "AD"])
].copy()
healthy_amm = df_missing_matched_age_matched[
    df_missing_matched_age_matched["Status"] == "healthy"
].copy()

sick_amm["_feat"] = get_missing_features(sick_amm, columns_with_missing_values)
healthy_amm["_feat"] = get_missing_features(healthy_amm, columns_with_missing_values)

matched_healthy_amm_list = []
used_eids = set()

for (age, feat), group in sick_amm.groupby(["Age", "_feat"]):
    count = len(group)
    needed = count * 10 if len(feat) == 0 else count * 1
    candidates = healthy_amm[
        (healthy_amm["_feat"] == feat)
        & (healthy_amm["Age"] == age)
        & (~healthy_amm["eid"].isin(used_eids))
    ]

    # if no candidate for this age group do + - 2 years
    if len(candidates) == 0:
        candidates = healthy_amm[
            (healthy_amm["_feat"] == feat)
            & (healthy_amm["Age"] <= age + 2)
            & (healthy_amm["Age"] >= age - 2)
            & (~healthy_amm["eid"].isin(used_eids))
        ]

    if len(candidates) >= needed:
        sampled = candidates.sample(n=needed, random_state=41 + SEED)
    else:
        sampled = candidates
        print(
            f"Warning (age+missing-matched): Only {len(candidates)} healthy candidates "
            f"for age between {age - 2} - {age + 2}, pattern {list(feat) if feat else 'no-missing'}, needed {needed}"
        )
    used_eids.update(sampled["eid"].tolist())
    matched_healthy_amm_list.append(sampled)

matched_healthy_amm = pd.concat(matched_healthy_amm_list, ignore_index=True)


# drop helper column
sick_amm = sick_amm.drop(columns=["_feat"])
matched_healthy_amm = matched_healthy_amm.drop(columns=["_feat"])

# combine & shuffle
df_missing_matched_age = pd.concat([sick_amm, matched_healthy_amm], ignore_index=True)
df_missing_matched_age = df_missing_matched_age.sample(
    frac=1, random_state=41 + SEED
).reset_index(drop=True)

print("\nAge + Missing-Matched dataset distribution:")
print(df_missing_matched_age["Status"].value_counts())

In [ ]:
# imputation pipelines
df_amm_missing = df_missing_matched_age.copy()
df_amm_randomfill = df_missing_matched_age.copy()
df_amm_knn = df_missing_matched_age.copy()

df_amm_missing.to_csv(
    output_path + r"missing_matched_age_matched_missing.csv", index=False
)

df_amm_randomfill = randomfill_missing_values(
    df_amm_randomfill, columns_with_missing_values
)
df_amm_randomfill.to_csv(
    output_path + r"missing_matched_age_matched_randomfill.csv", index=False
)

df_amm_knn = custom_knn_impute_euclidean(
    df_amm_knn,
    k=5,
    NUM_FEATURES=NUM_FEATURES,
    categorical_cols=CAT_FEATURES,
    columns_with_missing_values=columns_with_missing_values,
    distance_func=cosine_similarity,
)
df_amm_knn.to_csv(output_path + r"missing_matched_age_matched_knn.csv", index=False)